# 01 · Ingestão fonte → RAW — 4 abordagens

Trazer Postgres → camada RAW (Delta), comparando as formas de fazer. Objetivo:
achar o que **não estoura a RAM** e escala pra 100M linhas. Resultados e números em
[RESULTADOS.md](RESULTADOS.md).

| # | Abordagem | Ponto |
|---|---|---|
| A | Caseiro (lista + `write_delta`) | 🛑 acumula tudo na RAM → OOM em 1–2M |
| B | Batch (flush em lotes) | memória O(batch), mas row-by-row em Python |
| C | dlt → DuckDB | streaming gerenciado, destino idiomático |
| D | dlt + Polars (backend `connectorx`) | ✅ escolhido — Rust, N conexões, Delta nativo |

In [ ]:
import sys; sys.path.insert(0, "/Users/allanfraga/Repos/strattum/experimentacoes")
import exp
TABLE = "customers"   # troque por customers_5m / _10m / _100m pra escalar

## A · Caseiro — o código atual da plataforma
Gerador → `append` numa lista → `write_delta` no fim. **Segura a tabela inteira na RAM.**

In [ ]:
import uuid
from connectors.postgres.connector import PostgresConnector
from connectors.utils.delta import write_delta

conn = PostgresConnector(); conn.authenticate({"connection_string": exp.dsn()})
with exp.measure(f"caseiro {TABLE}"):
    rows = []
    for record in conn.extract(f"public__{TABLE}", state={"mode": "full_refresh"}):
        record["_dlt_id"] = str(uuid.uuid4())
        rows.append(record)                       # 🛑 acumula tudo na RAM
    write_delta(rows, f"{exp.DATA}/raw/postgres/public/{TABLE}", mode="overwrite")
print("linhas:", len(rows))

## B · Batch — flush a cada N
Mesmo extract do caseiro, mas grava em lotes → memória O(batch). Ainda row-by-row (GIL).

In [ ]:
import uuid
from connectors.postgres.connector import PostgresConnector
from connectors.utils.delta import write_delta

BATCH = 50_000
path  = f"{exp.DATA}/raw_batch/postgres/public/{TABLE}"
conn = PostgresConnector(); conn.authenticate({"connection_string": exp.dsn()})
with exp.measure(f"batch {TABLE} (n={BATCH})"):
    buffer, first, total = [], True, 0
    for record in conn.extract(f"public__{TABLE}", state={"mode": "full_refresh"}):
        record["_dlt_id"] = str(uuid.uuid4())
        buffer.append(record)
        if len(buffer) >= BATCH:
            write_delta(buffer, path, mode="overwrite" if first else "append")
            total, first = total + len(buffer), False; buffer.clear()
    if buffer:
        write_delta(buffer, path, mode="overwrite" if first else "append"); total += len(buffer)
print("linhas:", total)

## C · dlt → DuckDB
dlt gerencia memória via streaming (`pipeline.run()`); destino idiomático DuckDB.

In [ ]:
from dlt.sources.sql_database import sql_database

source = sql_database(credentials=exp.dsn(), schema="public",
                      table_names=[TABLE], backend="pyarrow")
source.resources[TABLE].apply_hints(write_disposition="replace")
pipeline = exp.dlt_pipeline()                     # estado limpo + destino DuckDB
with exp.measure(f"dlt {TABLE}"):
    pipeline.run(source)
print("linhas:", exp.count_duckdb(exp.DUCKDB, f"postgres.{TABLE}"))

## D · dlt + Polars 💡 (a escolha)
dlt traz a amplitude de conectores; Polars transforma e grava **Delta nativo**.
Compara os backends de leitura: **`pyarrow`** (1 conexão) vs **`connectorx`** (N conexões, Rust).
`connectorx` vence — é o que a plataforma adota (ver [resultados §1](../../docs/arquitetura/2.0-lake-aberto/descobertas.md)).

In [ ]:
import polars as pl
from dlt.sources.sql_database import sql_table

def run(backend):
    path   = f"{exp.DATA}/raw_dltpolars/{backend}/{TABLE}"
    source = sql_table(credentials=exp.dsn(), schema="public", table=TABLE, backend=backend)
    with exp.measure(f"dlt+polars [{backend}] {TABLE}"):
        df = pl.concat([pl.from_arrow(chunk) for chunk in source], how="vertical_relaxed")
        df = df.with_columns(pl.col("email").str.strip_chars().str.to_lowercase())   # transform
        df.write_delta(path, mode="overwrite")                                       # Delta nativo
    print("linhas:", exp.count_delta(path))

run("pyarrow")
run("connectorx")